In [27]:
import pandas as pd
import numpy as np

In [28]:
df = pd.read_csv('Dataset/raw.csv')
df.head()

,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxND...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,/Apple-Cancellation-Transparency-Personalized-...,2025-08-21 11:14:29
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref...,2025-08-21 11:14:29


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   title                     42675 non-null  object
 1   rating                    41651 non-null  object
 2   number_of_reviews         41651 non-null  object
 3   bought_in_last_month      39458 non-null  object
 4   current/discounted_price  30926 non-null  object
 5   price_on_variant          42675 non-null  object
 6   listed_price              42675 non-null  object
 7   is_best_seller            42675 non-null  object
 8   is_sponsored              42675 non-null  object
 9   is_couponed               42675 non-null  object
 10  buy_box_availability      28022 non-null  object
 11  delivery_details          30955 non-null  object
 12  sustainability_badges     3408 non-null   object
 13  image_url                 42675 non-null  object
 14  product_url           

In [30]:
print(df.isnull().sum())

title                           0
rating                       1024
number_of_reviews            1024
bought_in_last_month         3217
current/discounted_price    11749
price_on_variant                0
listed_price                    0
is_best_seller                  0
is_sponsored                    0
is_couponed                     0
buy_box_availability        14653
delivery_details            11720
sustainability_badges       39267
image_url                       0
product_url                  2069
collected_at                    0
dtype: int64


In [31]:
# Drop unrelated columns
columns_to_drop = ['delivery_details','sustainability_badges','image_url','collected_at']

df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

In [32]:
# Rename columns for clarity
df.rename(columns={'title':'product_title',
                   'rating':'product_rating',
                   'number_of_reviews':'total_reviews',
                   'bought_in_last_month':'purchased_last_month',
                   'current/discounted_price':'discounted_price',
                   'price_on_variant':'variant_price',
                   'is_couponed':'has_couponed'
                   }, inplace=True)

In [33]:
df.duplicated().sum()

np.int64(1839)

In [34]:
# Remove duplicate
df.drop_duplicates(inplace=True)

In [35]:
# Remove product_url after checking duplicate
df.drop(columns=['product_url'], inplace=True, errors='ignore')

Handle wrong data types

In [36]:
# prodcut_rating
df['product_rating'] = df['product_rating'].str.extract(r'([0-9.]+)').astype(float)

In [37]:
# total_reviews
df['total_reviews'] = df['total_reviews'].str.replace(',','')
df['total_reviews'] = df['total_reviews'].fillna(0)
df['total_reviews'] = df['total_reviews'].astype(int)

In [38]:
import re

def clean_bought_volume(value):
    val_str = str(value).strip()
    
    if val_str == 'nan' or 'bought' not in val_str:
        if '(' in val_str or 'Price' in val_str or 'Rating' in val_str:
            return None 
        return 0 
        
    match = re.search(r'(\d+)([KK]?)', val_str)
    if match:
        number = int(match.group(1))
        unit = match.group(2).upper()
        
        if unit == 'K':
            return number * 1000
        return number
        
    return 0

df['purchased_last_month'] = df['purchased_last_month'].apply(clean_bought_volume)


In [39]:
# is_best_seller
df['is_best_seller'] = df['is_best_seller'] == "Best Seller"

In [40]:
# buy_box_availability
df['buy_box_availability'] = df['buy_box_availability'].fillna("Unavailable/Out of Stock")

In [41]:
# handle price columns
df['extracted_variant_price'] = df['variant_price'].str.extract(r'\$(\d+\.\d+)').astype(float)

df['final_price'] = df['discounted_price']

df['final_price'] = df['final_price'].fillna(df['extracted_variant_price'])

In [42]:
df['final_price'] = df['final_price'].astype(str).str.replace(',', '')
df['final_price'] = df['final_price'].astype(float)

In [43]:
# is_sponsored column
df['is_sponsored'] = df['is_sponsored'] == "Sponsored"

In [44]:
# If discounted_price is NULl -> True, else -> False
df['is_discounted'] = df['discounted_price'].notna()

In [45]:
# Drop null value for final_price column
df.dropna(subset=['final_price'], inplace=True)

columns_to_drop = ['discounted_price', 'variant_price', 'listed_price', 'extracted_variant_price']
df.drop(columns=columns_to_drop, inplace=True)

In [46]:
# has_couponed
df['has_couponed']=df['has_couponed'] != "No Coupon"

In [47]:
# Handdle missing values for product_rating and purchased_last_month
df['product_rating'] = df['product_rating'].fillna(0)

df.dropna(subset=['purchased_last_month'], inplace=True)


In [48]:
# Create Category column from title
import pandas as pd

def assign_category(title):
    # Chuyển về chữ thường để không phân biệt HOA - thường
    title = str(title).lower()
    
    # Định nghĩa bộ từ khóa nhận diện cho từng nhóm
    if 'charger' in title or 'cable' in title or 'adapter' in title:
        return 'Chargers & Cables' 
    elif"microphone" in title or 'headphone' in title or 'earbud' in title or 'earphone' in title:
        return 'Headphones' 
    elif 'router' in title or 'switch' in title or 'wifi' in title or 'networking' in title:
        return 'Networking'
    elif 'battery' in title or 'power bank' in title:
        return 'Power & Batteries'
    elif "micro" in title or "SD" in title or 'ssd' in title or 'hdd' in title or 'hard drive' in title or 'flash drive' in title or 'storage' in title:
        return 'Storage'
    elif 'speaker' in title or 'soundbar' in title:
        return 'Speakers'
    elif 'printer' in title or 'scanner' in title:
        return 'Printers & Scanners'
    elif 'phone' in title or 'iphone' in title or 'samsung' in title: # Loại trừ headphone đã bắt ở trên
        return 'Phones'
    elif 'laptop' in title or 'macbook' in title or 'notebook' in title:
        return 'Laptops'
    elif 'camera' in title or 'webcam' in title or 'gimbal' in title:
        return 'Cameras'
    elif 'tv' in title or 'monitor' in title or 'display' in title:
        return 'TV & Display'
    elif 'watch' in title or 'smartwatch' in title or 'wearable' in title:
        return 'Wearables'
    elif 'smart home' in title or 'alexa' in title or 'echo' in title:
        return 'Smart Home'
    elif 'game' in title or 'playstation' in title or 'xbox' in title or 'nintendo' in title:
        return 'Gaming'
    
    # Nhóm mặc định nếu không khớp từ khóa nào ở trên
    return 'Other Electronics'

# Áp dụng hàm để tạo cột mới
df['Category'] = df['product_title'].apply(assign_category)

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37474 entries, 0 to 42674
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_title         37474 non-null  object 
 1   product_rating        37474 non-null  float64
 2   total_reviews         37474 non-null  int64  
 3   purchased_last_month  37474 non-null  float64
 4   is_best_seller        37474 non-null  bool   
 5   is_sponsored          37474 non-null  bool   
 6   has_couponed          37474 non-null  bool   
 7   buy_box_availability  37474 non-null  object 
 8   final_price           37474 non-null  float64
 9   is_discounted         37474 non-null  bool   
 10  Category              37474 non-null  object 
dtypes: bool(4), float64(3), int64(1), object(3)
memory usage: 2.4+ MB


In [50]:
df.head()

,product_title,product_rating,total_reviews,purchased_last_month,is_best_seller,is_sponsored,has_couponed,buy_box_availability,final_price,is_discounted,Category
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375,300.0,False,True,True,Add to cart,89.68,True,Headphones
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457,6000.0,False,True,False,Add to cart,9.99,True,Chargers & Cables
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044,2000.0,False,True,False,Add to cart,314.00,True,Headphones
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882,10000.0,True,False,False,Unavailable/Out of Stock,162.24,False,Headphones
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988,10000.0,False,False,False,Unavailable/Out of Stock,72.74,False,Phones


In [52]:
df.to_csv('Dataset/cleaned_data.csv', index=False)